# Stint Construction

This notebook reconstructs lineup stints from play-by-play data for a single NBA game.

A stint is defined as a continuous stretch of game time during which the ten players on the floor remain unchanged. Each substitution marks the end of one stint and the beginning of the next.

The purpose of this notebook is to build the first working version of a lineup-aware event pipeline that can later be generalized across games and seasons.

## Why this step matters

Stint construction is the core engineering layer of the project.

Raw play-by-play data records individual events, but lineup analytics requires a different unit of analysis: stable on-court player combinations over time. In order to evaluate player impact and lineup performance, the event stream must be transformed into possession-aware game segments where both team lineups are known.

This notebook creates that first prototype.

## Objectives

By the end of this notebook, we want to:

- isolate one full game from the play-by-play table
- clean and standardize IDs
- parse score progression into numeric fields
- identify the two teams in the game
- extract substitutions in structured form
- build a robust player-name lookup
- infer and validate the starting lineups
- track lineup changes over time
- segment the game into stints
- create a clean stint-level dataset for downstream analysis

In [30]:
import sqlite3
import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 200)

DB_PATH = "../data/raw/nba.sqlite"
conn = sqlite3.connect(DB_PATH)

## Load one prototype game

The first version of the stint engine is developed on a single game. This makes it easier to validate assumptions, inspect lineup transitions, and debug the event loop before scaling to larger datasets.

In [31]:
sample_game_id = "0011300001"

game_raw = pd.read_sql(
    f"""
    SELECT *
    FROM play_by_play
    WHERE game_id = '{sample_game_id}'
    ORDER BY period, eventnum
    """,
    conn
)

game_raw.head(10)

,game_id,eventnum,eventmsgtype,eventmsgactiontype,period,wctimestring,pctimestring,homedescription,neutraldescription,visitordescription,score,scoremargin,person1type,player1_id,player1_name,player1_team_id,player1_team_city,player1_team_nickname,player1_team_abbreviation,person2type,player2_id,player2_name,player2_team_id,player2_team_city,player2_team_nickname,player2_team_abbreviation,person3type,player3_id,player3_name,player3_team_id,player3_team_city,player3_team_nickname,player3_team_abbreviation,video_available_flag
0,0011300001,2,12,0,1,1:04 PM,12:00,None,Start of 1st Period (1:04 PM EST),None,None,None,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
1,0011300001,3,10,0,1,1:05 PM,12:00,Jump Ball Zoric vs. Ibaka: Tip to Jackson,None,None,None,None,4.0,42540,Luka Zoric,12321.0,Istanbul,Fenerbahce Ulker,FBU,5.0,201586,Serge Ibaka,1610612760.0,Oklahoma City,Thunder,OKC,5.0,202704,Reggie Jackson,1610612760.0,Oklahoma City,Thunder,OKC,0
2,0011300001,4,1,1,1,1:05 PM,11:46,None,None,Ibaka 17' Jump Shot (2 PTS) (Jackson 1 AST),2 - 0,-2,5.0,201586,Serge Ibaka,1610612760.0,Oklahoma City,Thunder,OKC,5.0,202704,Reggie Jackson,1610612760.0,Oklahoma City,Thunder,OKC,0.0,0,None,None,None,None,None,0
3,0011300001,5,1,46,1,1:06 PM,11:31,Preldzic 7' Running Jump Shot (2 PTS),None,None,2 - 2,TIE,4.0,42545,Emir Preldzic,12321.0,Istanbul,Fenerbahce Ulker,FBU,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
4,0011300001,6,6,2,1,1:06 PM,11:31,None,None,Durant S.FOUL (P1.T1),None,None,5.0,201142,Kevin Durant,1610612760.0,Oklahoma City,Thunder,OKC,4.0,42545,Emir Preldzic,12321.0,Istanbul,Fenerbahce Ulker,FBU,1.0,0,None,None,None,None,None,0
5,0011300001,7,3,10,1,1:06 PM,11:31,MISS Preldzic Free Throw 1 of 1,None,None,None,None,4.0,42545,Emir Preldzic,12321.0,Istanbul,Fenerbahce Ulker,FBU,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
6,0011300001,9,4,0,1,1:06 PM,11:29,None,None,Jackson REBOUND (Off:0 Def:1),None,None,5.0,202704,Reggie Jackson,1610612760.0,Oklahoma City,Thunder,OKC,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
7,0011300001,10,2,1,1,1:06 PM,11:15,None,None,MISS Durant 13' Jump Shot,None,None,5.0,201142,Kevin Durant,1610612760.0,Oklahoma City,Thunder,OKC,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
8,0011300001,11,4,0,1,1:07 PM,11:14,None,None,Perkins REBOUND (Off:1 Def:0),None,None,5.0,2570,Kendrick Perkins,1610612760.0,Oklahoma City,Thunder,OKC,0.0,0,None,None,None,None,None,0.0,0,None,None,None,None,None,0
9,0011300001,12,6,1,1,1:07 PM,11:14,Zoric P.FOUL (P1.T1),None,None,None,None,4.0,42540,Luka Zoric,12321.0,Istanbul,Fenerbahce Ulker,FBU,5.0,2570,Kendrick Perkins,1610612760.0,Oklahoma City,Thunder,OKC,1.0,0,None,None,None,None,None,0


## Keep only the columns needed for lineup reconstruction

The raw event table contains more fields than are needed for the prototype. This section creates a focused event dataset containing only the variables required for:

- chronology
- score progression
- substitution handling
- player/team identification

In [32]:
cols_needed = [
    "game_id",
    "eventnum",
    "eventmsgtype",
    "eventmsgactiontype",
    "period",
    "wctimestring",
    "pctimestring",
    "homedescription",
    "neutraldescription",
    "visitordescription",
    "score",
    "scoremargin",
    "player1_id",
    "player1_name",
    "player1_team_id",
    "player2_id",
    "player2_name",
    "player2_team_id",
    "player3_id",
    "player3_name",
    "player3_team_id",
]

game = game_raw[cols_needed].copy()
game.head(10)

,game_id,eventnum,eventmsgtype,eventmsgactiontype,period,wctimestring,pctimestring,homedescription,neutraldescription,visitordescription,score,scoremargin,player1_id,player1_name,player1_team_id,player2_id,player2_name,player2_team_id,player3_id,player3_name,player3_team_id
0,0011300001,2,12,0,1,1:04 PM,12:00,None,Start of 1st Period (1:04 PM EST),None,None,None,0,None,None,0,None,None,0,None,None
1,0011300001,3,10,0,1,1:05 PM,12:00,Jump Ball Zoric vs. Ibaka: Tip to Jackson,None,None,None,None,42540,Luka Zoric,12321.0,201586,Serge Ibaka,1610612760.0,202704,Reggie Jackson,1610612760.0
2,0011300001,4,1,1,1,1:05 PM,11:46,None,None,Ibaka 17' Jump Shot (2 PTS) (Jackson 1 AST),2 - 0,-2,201586,Serge Ibaka,1610612760.0,202704,Reggie Jackson,1610612760.0,0,None,None
3,0011300001,5,1,46,1,1:06 PM,11:31,Preldzic 7' Running Jump Shot (2 PTS),None,None,2 - 2,TIE,42545,Emir Preldzic,12321.0,0,None,None,0,None,None
4,0011300001,6,6,2,1,1:06 PM,11:31,None,None,Durant S.FOUL (P1.T1),None,None,201142,Kevin Durant,1610612760.0,42545,Emir Preldzic,12321.0,0,None,None
5,0011300001,7,3,10,1,1:06 PM,11:31,MISS Preldzic Free Throw 1 of 1,None,None,None,None,42545,Emir Preldzic,12321.0,0,None,None,0,None,None
6,0011300001,9,4,0,1,1:06 PM,11:29,None,None,Jackson REBOUND (Off:0 Def:1),None,None,202704,Reggie Jackson,1610612760.0,0,None,None,0,None,None
7,0011300001,10,2,1,1,1:06 PM,11:15,None,None,MISS Durant 13' Jump Shot,None,None,201142,Kevin Durant,1610612760.0,0,None,None,0,None,None
8,0011300001,11,4,0,1,1:07 PM,11:14,None,None,Perkins REBOUND (Off:1 Def:0),None,None,2570,Kendrick Perkins,1610612760.0,0,None,None,0,None,None
9,0011300001,12,6,1,1,1:07 PM,11:14,Zoric P.FOUL (P1.T1),None,None,None,None,42540,Luka Zoric,12321.0,2570,Kendrick Perkins,1610612760.0,0,None,None


##  Standardize identifiers

Some player and team ID fields appear as floats when read into pandas. To avoid downstream issues, all player and team identifiers are standardized as strings.

In [33]:
def clean_id(x):
    if pd.isna(x):
        return np.nan
    try:
        return str(int(float(x)))
    except:
        return str(x)

id_cols = [
    "player1_id", "player1_team_id",
    "player2_id", "player2_team_id",
    "player3_id", "player3_team_id"
]

for col in id_cols:
    game[col] = game[col].apply(clean_id)

game.head(5)

,game_id,eventnum,eventmsgtype,eventmsgactiontype,period,wctimestring,pctimestring,homedescription,neutraldescription,visitordescription,score,scoremargin,player1_id,player1_name,player1_team_id,player2_id,player2_name,player2_team_id,player3_id,player3_name,player3_team_id
0,0011300001,2,12,0,1,1:04 PM,12:00,None,Start of 1st Period (1:04 PM EST),None,None,None,0,None,NaN,0,None,NaN,0,None,NaN
1,0011300001,3,10,0,1,1:05 PM,12:00,Jump Ball Zoric vs. Ibaka: Tip to Jackson,None,None,None,None,42540,Luka Zoric,12321,201586,Serge Ibaka,1610612760,202704,Reggie Jackson,1610612760
2,0011300001,4,1,1,1,1:05 PM,11:46,None,None,Ibaka 17' Jump Shot (2 PTS) (Jackson 1 AST),2 - 0,-2,201586,Serge Ibaka,1610612760,202704,Reggie Jackson,1610612760,0,None,NaN
3,0011300001,5,1,46,1,1:06 PM,11:31,Preldzic 7' Running Jump Shot (2 PTS),None,None,2 - 2,TIE,42545,Emir Preldzic,12321,0,None,NaN,0,None,NaN
4,0011300001,6,6,2,1,1:06 PM,11:31,None,None,Durant S.FOUL (P1.T1),None,None,201142,Kevin Durant,1610612760,42545,Emir Preldzic,12321,0,None,NaN


##  Parse score into numeric away and home score columns

The raw `score` field is stored as text. For stint-level analysis, it is more useful to work with explicit numeric score fields.

In [34]:
def parse_score(score_str):
    if pd.isna(score_str):
        return (np.nan, np.nan)
    
    parts = str(score_str).split(" - ")
    if len(parts) != 2:
        return (np.nan, np.nan)
    
    try:
        away_score = int(parts[0].strip())
        home_score = int(parts[1].strip())
        return (away_score, home_score)
    except:
        return (np.nan, np.nan)

game[["away_score", "home_score"]] = game["score"].apply(
    lambda x: pd.Series(parse_score(x))
)

game["away_score"] = game["away_score"].ffill().fillna(0).astype(int)
game["home_score"] = game["home_score"].ffill().fillna(0).astype(int)

game[["eventnum", "period", "pctimestring", "score", "away_score", "home_score"]].head(15)

,eventnum,period,pctimestring,score,away_score,home_score
0,2,1,12:00,None,0,0
1,3,1,12:00,None,0,0
2,4,1,11:46,2 - 0,2,0
3,5,1,11:31,2 - 2,2,2
4,6,1,11:31,None,2,2
5,7,1,11:31,None,2,2
6,9,1,11:29,None,2,2
7,10,1,11:15,None,2,2
8,11,1,11:14,None,2,2
9,12,1,11:14,None,2,2


## Score format note

For this dataset, the `score` field is interpreted as:

- first value = away score
- second value = home score

This is consistent with the event descriptions observed in earlier exploration.

## Identify the two teams in the game

Before tracking lineups, we need to identify the home and away team IDs. This is done in two steps:

1. confirm that only two unique team IDs appear in the game
2. infer which team is home and which is away using the description fields

In [35]:
team_ids = pd.concat([
    game["player1_team_id"],
    game["player2_team_id"],
    game["player3_team_id"]
]).dropna().unique().tolist()

team_ids

['12321', '1610612760']

In [36]:
home_team_candidates = game.loc[
    game["homedescription"].notna() & game["player1_team_id"].notna(),
    "player1_team_id"
]

away_team_candidates = game.loc[
    game["visitordescription"].notna() & game["player1_team_id"].notna(),
    "player1_team_id"
]

home_team_id = home_team_candidates.mode().iloc[0]
away_team_id = away_team_candidates.mode().iloc[0]

home_team_id, away_team_id

('12321', '1610612760')

## Team identification takeaway

The event stream contains exactly two team IDs, and the home/away assignment can be inferred from event descriptions. These team IDs will be used to update the correct lineup whenever a substitution occurs.

## Extract structured substitution events

Substitutions define the boundaries between lineup stints.

Based on the previous notebook, substitution rows satisfy:

- `eventmsgtype = 8`
- `player1_id` = player leaving the floor
- `player2_id` = player entering the floor
- `player1_team_id` = team making the substitution

In [37]:
subs = game[game["eventmsgtype"] == 8].copy()

subs["player_out_id"] = subs["player1_id"]
subs["player_out_name"] = subs["player1_name"]
subs["player_in_id"] = subs["player2_id"]
subs["player_in_name"] = subs["player2_name"]
subs["sub_team_id"] = subs["player1_team_id"]

subs[[
    "eventnum",
    "period",
    "pctimestring",
    "sub_team_id",
    "player_out_id",
    "player_out_name",
    "player_in_id",
    "player_in_name",
    "homedescription",
    "visitordescription"
]].head(15)

,eventnum,period,pctimestring,sub_team_id,player_out_id,player_out_name,player_in_id,player_in_name,homedescription,visitordescription
38,44,1,7:35,12321,42540,Luka Zoric,42538,Gasper Vidmar,SUB: Vidmar FOR Zoric,None
60,69,1,4:35,1610612760,2570,Kendrick Perkins,2555,Nick Collison,None,SUB: Collison FOR Perkins
66,77,1,3:57,1610612760,201142,Kevin Durant,203087,Jeremy Lamb,None,SUB: Lamb FOR Durant
69,81,1,3:57,12321,42545,Emir Preldzic,42535,Melih Mahmutoglu,SUB: Mahmutoglu FOR Preldzic,None
77,92,1,3:09,12321,42531,Bo McCalebb,42542,Kenan Sipahi,SUB: Sipahi FOR McCalebb,None
84,103,1,2:13,1610612760,201586,Serge Ibaka,201934,Hasheem Thabeet,None,SUB: Thabeet FOR Ibaka
93,113,1,1:28,12321,42536,None,42537,Izzet Turkyilmaz,SUB: Turkyilmaz FOR Kleiza,None
107,130,1,0:21,12321,42544,Bojan Bogdanovic,42541,James Birsen,SUB: Birsen FOR Bogdanovic,None
127,158,2,10:32,12321,42538,Gasper Vidmar,42540,Luka Zoric,SUB: Zoric FOR Vidmar,None
143,177,2,8:20,12321,42537,Izzet Turkyilmaz,42545,Emir Preldzic,SUB: Preldzic FOR Turkyilmaz,None


## Build the best available player-name lookup

The player reference table does not fully cover every player ID appearing in this game's event stream. To maximize name coverage, the player-name lookup is built in two layers:

1. names from the `player` table
2. names from the play-by-play table (`player1`, `player2`, `player3`)

This provides strong prototype-level coverage while preserving player IDs as the source of truth for lineup logic.

In [38]:
# Base lookup from player table
player_lookup = pd.read_sql(
    """
    SELECT DISTINCT id, full_name
    FROM player
    """,
    conn
)

player_lookup["id"] = player_lookup["id"].astype(str)

player_table_name_map = dict(
    zip(player_lookup["id"], player_lookup["full_name"])
)

# Additional lookup from play-by-play
pbp_player_lookup_1 = game[["player1_id", "player1_name"]].dropna().rename(
    columns={"player1_id": "id", "player1_name": "full_name"}
)

pbp_player_lookup_2 = game[["player2_id", "player2_name"]].dropna().rename(
    columns={"player2_id": "id", "player2_name": "full_name"}
)

pbp_player_lookup_3 = game[["player3_id", "player3_name"]].dropna().rename(
    columns={"player3_id": "id", "player3_name": "full_name"}
)

pbp_player_lookup = pd.concat(
    [pbp_player_lookup_1, pbp_player_lookup_2, pbp_player_lookup_3],
    axis=0
).drop_duplicates()

pbp_player_lookup["id"] = pbp_player_lookup["id"].astype(str)

pbp_name_map = dict(
    zip(pbp_player_lookup["id"], pbp_player_lookup["full_name"])
)

# Final combined lookup: player table first, pbp fallback / enrichment
player_name_map = {}
player_name_map.update(player_table_name_map)
player_name_map.update(pbp_name_map)

len(player_name_map)

4827

In [39]:
def map_player_names(player_ids):
    return [player_name_map.get(str(pid), str(pid)) for pid in player_ids]

## Name-mapping note

A small number of players may still remain unresolved if their IDs are present in the event stream but their names do not appear in either the player table or the event-level name fields. This does not affect lineup reconstruction, because all lineup logic is based on player IDs.

For the prototype game, any remaining missing names can be patched manually if needed for readability.

## Infer candidate starters for period 1

Because substitutions identify lineup changes but not the initial on-court state, we estimate candidate starters using two signals:

- players involved in early period 1 events before the first substitution
- players who are subbed out during period 1, since they must have been on the floor before leaving

In [40]:
def unique_players_from_events(df, team_id):
    players = set()

    for player_col, team_col in [
        ("player1_id", "player1_team_id"),
        ("player2_id", "player2_team_id"),
        ("player3_id", "player3_team_id")
    ]:
        temp = df.loc[
            df[team_col].astype(str) == str(team_id),
            player_col
        ].dropna().astype(str).tolist()

        players.update(temp)

    return players

# Period 1 substitutions
p1_subs = subs[subs["period"] == 1].copy()

players_subbed_out_p1 = (
    p1_subs.groupby("sub_team_id")["player_out_id"]
    .apply(list)
    .to_dict()
)

# Early window before first substitution
first_sub_eventnum = subs["eventnum"].min()

early_events = game[
    (game["period"] == 1) &
    (game["eventnum"] < first_sub_eventnum)
].copy()

# Early-event players
home_early_players = unique_players_from_events(early_events, home_team_id)
away_early_players = unique_players_from_events(early_events, away_team_id)

# Players subbed out in period 1
home_subbed_out = set(map(str, players_subbed_out_p1.get(home_team_id, [])))
away_subbed_out = set(map(str, players_subbed_out_p1.get(away_team_id, [])))

# Candidate starters
home_candidate_starters = home_early_players.union(home_subbed_out)
away_candidate_starters = away_early_players.union(away_subbed_out)

home_candidate_starters, away_candidate_starters

({'42531', '42536', '42540', '42544', '42545'},
 {'200757', '201142', '201586', '202704', '2570'})

In [41]:
print("Home candidate starters:", map_player_names(home_candidate_starters))
print("Away candidate starters:", map_player_names(away_candidate_starters))

Home candidate starters: ['Emir Preldzic', 'Bo McCalebb', '42536', 'Luka Zoric', 'Bojan Bogdanovic']
Away candidate starters: ['Kendrick Perkins', 'Reggie Jackson', 'Serge Ibaka', 'Thabo Sefolosha', 'Kevin Durant']


## Validate prototype starting lineups

Starter inference from play-by-play data can be noisy, so the first one-game version uses a transparent validation step: candidate starters are reviewed manually before constructing stints.

This ensures that the lineup-tracking logic is tested on a correct starting state before the starter inference method is generalized.

In [46]:
sample_game_info = pd.read_sql(
    f"""
    SELECT *
    FROM game
    WHERE game_id = '{sample_game_id}'
    """,
    conn
)

sample_game_info

,season_id,team_id_home,team_abbreviation_home,team_name_home,game_id,game_date,matchup_home,wl_home,min,fgm_home,fga_home,fg_pct_home,fg3m_home,fg3a_home,fg3_pct_home,ftm_home,fta_home,ft_pct_home,oreb_home,dreb_home,reb_home,ast_home,stl_home,blk_home,tov_home,pf_home,pts_home,plus_minus_home,video_available_home,team_id_away,team_abbreviation_away,team_name_away,matchup_away,wl_away,fgm_away,fga_away,fg_pct_away,fg3m_away,fg3a_away,fg3_pct_away,ftm_away,fta_away,ft_pct_away,oreb_away,dreb_away,reb_away,ast_away,stl_away,blk_away,tov_away,pf_away,pts_away,plus_minus_away,video_available_away,season_type
0,12013,12321,FBU,Istanbul Fenerbahce Ulker,0011300001,2013-10-05 00:00:00,FBU vs. OKC,L,240,26.0,68.0,0.382,10.0,28.0,0.357,20.0,31.0,0.645,10.0,17.0,27.0,13.0,11.0,2.0,16.0,24.0,82.0,-13,0,1610612760,OKC,Oklahoma City Thunder,OKC @ FBU,W,36.0,74.0,0.486,2.0,14.0,0.143,21.0,28.0,0.75,18.0,34.0,52.0,22.0,9.0,8.0,20.0,26.0,95.0,13,0,Pre Season


In [49]:
# Optional manual patches for unresolved names in this prototype game
# Example:
player_name_map["42536"] = "Linas Kleiza"

In [50]:
# Final validated starters for the prototype game
home_starters = ["42531", "42536", "42540", "42544", "42545"]
away_starters = ["200757", "201142", "201586", "202704", "2570"]

print("Initial home lineup:", map_player_names(home_starters))
print("Initial away lineup:", map_player_names(away_starters))

Initial home lineup: ['Bo McCalebb', 'Linas Kleiza', 'Luka Zoric', 'Bojan Bogdanovic', 'Emir Preldzic']
Initial away lineup: ['Thabo Sefolosha', 'Kevin Durant', 'Serge Ibaka', 'Reggie Jackson', 'Kendrick Perkins']


## Data coverage note: international preseason matchup

This prototype game corresponds to a preseason matchup between an NBA team, the Oklahoma City Thunder, and a European club, Fenerbahçe Ülker.

Because the dataset is primarily designed for NBA games, player coverage in reference tables (such as the `player` table) is not guaranteed for all non-NBA participants. In addition, some players may not appear in play-by-play name fields if they are not directly involved in recorded events during the early stages of the game.

As a result, a small number of players may remain identified only by their IDs rather than full names. This does not affect the correctness of the lineup reconstruction, since all logic is based on player IDs.

This scenario highlights a common real-world challenge in sports analytics: integrating event data across leagues with different levels of standardization and coverage. Future iterations of this project may address this by incorporating additional player reference sources.

This design choice ensures that the analytical pipeline remains robust even when auxiliary data sources are incomplete.

##  Initialize the on-court lineup state

With both starting lineups validated, we can initialize the on-court state for the game. This state will be updated sequentially whenever a substitution event occurs.

In [51]:
current_home_lineup = set(map(str, home_starters))
current_away_lineup = set(map(str, away_starters))

current_home_lineup, current_away_lineup

({'42531', '42536', '42540', '42544', '42545'},
 {'200757', '201142', '201586', '202704', '2570'})

##  Build lineup stints

A new stint begins whenever the on-court state changes. In this prototype, stint boundaries are created at:

- the start of the game
- each substitution
- each period boundary

The loop below processes the event stream in order, closes the current stint at each boundary, updates the relevant lineup if needed, and starts the next stint with the new on-court state.

In [75]:
stints = []
sub_anomalies = []

# Re-initialize lineups before building stints
current_home_lineup = set(map(str, home_starters))
current_away_lineup = set(map(str, away_starters))

# Initialize first stint
first_row = game.iloc[0]

current_stint = {
    "game_id": sample_game_id,
    "period": int(first_row["period"]),
    "start_eventnum": int(first_row["eventnum"]),
    "start_clock": first_row["pctimestring"],
    "home_lineup": sorted(current_home_lineup),
    "away_lineup": sorted(current_away_lineup),
    "start_home_score": int(first_row["home_score"]),
    "start_away_score": int(first_row["away_score"])
}

for idx in range(len(game) - 1):
    row = game.iloc[idx]
    next_row = game.iloc[idx + 1]

    event_type = int(row["eventmsgtype"])
    eventnum = int(row["eventnum"])
    clock = row["pctimestring"]

    boundary = event_type in [8, 12, 13]

    if boundary:
        # Close current stint
        current_stint["end_eventnum"] = eventnum
        current_stint["end_clock"] = clock
        current_stint["end_home_score"] = int(row["home_score"])
        current_stint["end_away_score"] = int(row["away_score"])
        current_stint["home_points"] = (
            current_stint["end_home_score"] - current_stint["start_home_score"]
        )
        current_stint["away_points"] = (
            current_stint["end_away_score"] - current_stint["start_away_score"]
        )

        if current_stint["start_eventnum"] != current_stint["end_eventnum"]:
            stints.append(current_stint.copy())

        # Apply substitution with tolerant logic
        if event_type == 8:
            team_id = str(row["player1_team_id"])
            player_out = str(row["player1_id"])
            player_in = str(row["player2_id"])

            if team_id == str(home_team_id):
                lineup = current_home_lineup
                team_label = "home"
            elif team_id == str(away_team_id):
                lineup = current_away_lineup
                team_label = "away"
            else:
                lineup = None
                team_label = "unknown"

            if lineup is not None:
                before_lineup = sorted(lineup)

                # Try to remove outgoing player
                if player_out in lineup:
                    lineup.remove(player_out)
                    removed = True
                else:
                    removed = False

                # Try to add incoming player
                if player_in not in lineup:
                    lineup.add(player_in)
                    added = True
                else:
                    added = False

                # Safeguard: avoid dropping below 5
                if len(lineup) < 5 and removed and not added:
                    lineup.add(player_out)

                # Safeguard: avoid growing above 5
                # Prefer removing player_out if it was not actually removed and still remains,
                # otherwise trim using a stable deterministic rule.
                if len(lineup) > 5:
                    if player_out in lineup and player_in in lineup:
                        lineup.discard(player_out)

                    while len(lineup) > 5:
                        # deterministic fallback: remove the max string id
                        lineup.remove(sorted(lineup)[-1])

                after_lineup = sorted(lineup)

                if removed and added:
                    action_taken = "applied_clean"
                elif removed and not added:
                    action_taken = "removed_only"
                elif not removed and added:
                    action_taken = "added_only"
                else:
                    action_taken = "no_change"

                sub_anomalies.append({
                    "eventnum": int(row["eventnum"]),
                    "period": int(row["period"]),
                    "pctimestring": row["pctimestring"],
                    "team_label": team_label,
                    "team_id": team_id,
                    "player_out": player_out,
                    "player_in": player_in,
                    "removed": removed,
                    "added": added,
                    "lineup_size_after": len(lineup),
                    "action_taken": action_taken,
                    "before_lineup": before_lineup,
                    "after_lineup": after_lineup
                })

        # Start next stint from next row
        current_stint = {
            "game_id": sample_game_id,
            "period": int(next_row["period"]),
            "start_eventnum": int(next_row["eventnum"]),
            "start_clock": next_row["pctimestring"],
            "home_lineup": sorted(current_home_lineup),
            "away_lineup": sorted(current_away_lineup),
            "start_home_score": int(next_row["home_score"]),
            "start_away_score": int(next_row["away_score"])
        }

# Close final stint
last_row = game.iloc[-1]

current_stint["end_eventnum"] = int(last_row["eventnum"])
current_stint["end_clock"] = last_row["pctimestring"]
current_stint["end_home_score"] = int(last_row["home_score"])
current_stint["end_away_score"] = int(last_row["away_score"])
current_stint["home_points"] = (
    current_stint["end_home_score"] - current_stint["start_home_score"]
)
current_stint["away_points"] = (
    current_stint["end_away_score"] - current_stint["start_away_score"]
)

if current_stint["start_eventnum"] != current_stint["end_eventnum"]:
    stints.append(current_stint.copy())

# Convert outputs to dataframes
stints_df = pd.DataFrame(stints)
sub_anomalies_df = pd.DataFrame(sub_anomalies)

In [76]:
def lineup_ids_to_names(lineup_ids):
    return [player_name_map.get(str(pid), str(pid)) for pid in lineup_ids]

stints_df["home_lineup_names"] = stints_df["home_lineup"].apply(lineup_ids_to_names)
stints_df["away_lineup_names"] = stints_df["away_lineup"].apply(lineup_ids_to_names)

##  Sanity checks

Before using the stint dataset in later notebooks, it is important to verify that the lineup reconstruction behaves sensibly.

In [77]:
stints_df["home_n_players"] = stints_df["home_lineup"].apply(len)
stints_df["away_n_players"] = stints_df["away_lineup"].apply(len)

stints_df[["home_n_players", "away_n_players"]].value_counts()

home_n_players  away_n_players
5               5                 32
dtype: int64

In [78]:
sub_anomalies_df["action_taken"].value_counts()

applied_clean    29
no_change         3
added_only        2
removed_only      1
Name: action_taken, dtype: int64

## Sanity check interpretation

The stint reconstruction pipeline now produces consistent five-player lineups for both teams across all segments of the game. This confirms that the lineup state is stable and that the substitution-handling logic preserves roster integrity throughout the event stream.

Score progression across stints also behaves as expected, with point differentials aligning with the underlying play-by-play data. This indicates that stint boundaries and score tracking are correctly implemented.

While most substitutions are applied cleanly, a small number of events require tolerant handling due to inconsistencies between the play-by-play feed and the inferred on-court state. These cases are logged separately, allowing the model to remain stable without discarding valuable diagnostic information.

Overall, the prototype now provides a reliable foundation for downstream lineup analysis, while maintaining transparency around the remaining edge cases in the data.

## Substitution update note

Not every substitution event aligns perfectly with the inferred on-court state. To keep the prototype stable, substitutions are applied with a tolerant update rule and each event is logged for later inspection.

This allows the stint engine to preserve five-player lineup integrity while still documenting where the event stream and reconstructed state disagree.

##  Add lineup names for readability

Lineup reconstruction is driven by player IDs, but player names make the output much easier to inspect and validate.

In [79]:
def lineup_ids_to_names(lineup_ids):
    return [player_name_map.get(str(pid), str(pid)) for pid in lineup_ids]

stints_df["home_lineup_names"] = stints_df["home_lineup"].apply(lineup_ids_to_names)
stints_df["away_lineup_names"] = stints_df["away_lineup"].apply(lineup_ids_to_names)

stints_df[[
    "period",
    "start_eventnum",
    "end_eventnum",
    "start_clock",
    "end_clock",
    "home_lineup_names",
    "away_lineup_names",
    "home_points",
    "away_points"
]].head(10)

,period,start_eventnum,end_eventnum,start_clock,end_clock,home_lineup_names,away_lineup_names,home_points,away_points
0,1,3,44,12:00,7:35,"[Bo McCalebb, Linas Kleiza, Luka Zoric, Bojan Bogdanovic, Emir Preldzic]","[Thabo Sefolosha, Kevin Durant, Serge Ibaka, Reggie Jackson, Kendrick Perkins]",10,6
1,1,45,69,7:33,4:35,"[Bo McCalebb, Linas Kleiza, Gasper Vidmar, Bojan Bogdanovic, Emir Preldzic]","[Thabo Sefolosha, Kevin Durant, Serge Ibaka, Reggie Jackson, Kendrick Perkins]",10,6
2,1,70,77,4:21,3:57,"[Bo McCalebb, Linas Kleiza, Gasper Vidmar, Bojan Bogdanovic, Emir Preldzic]","[Thabo Sefolosha, Kevin Durant, Serge Ibaka, Reggie Jackson, Nick Collison]",0,3
3,1,78,81,3:57,3:57,"[Bo McCalebb, Linas Kleiza, Gasper Vidmar, Bojan Bogdanovic, Emir Preldzic]","[Thabo Sefolosha, Serge Ibaka, Reggie Jackson, Jeremy Lamb, Nick Collison]",1,0
4,1,82,92,3:57,3:09,"[Bo McCalebb, Melih Mahmutoglu, Linas Kleiza, Gasper Vidmar, Bojan Bogdanovic]","[Thabo Sefolosha, Serge Ibaka, Reggie Jackson, Jeremy Lamb, Nick Collison]",0,0
5,1,93,103,3:09,2:13,"[Melih Mahmutoglu, Linas Kleiza, Gasper Vidmar, Kenan Sipahi, Bojan Bogdanovic]","[Thabo Sefolosha, Serge Ibaka, Reggie Jackson, Jeremy Lamb, Nick Collison]",3,0
6,1,104,113,1:55,1:28,"[Melih Mahmutoglu, Linas Kleiza, Gasper Vidmar, Kenan Sipahi, Bojan Bogdanovic]","[Thabo Sefolosha, Hasheem Thabeet, Reggie Jackson, Jeremy Lamb, Nick Collison]",0,1
7,1,114,130,1:25,0:21,"[Melih Mahmutoglu, Izzet Turkyilmaz, Gasper Vidmar, Kenan Sipahi, Bojan Bogdanovic]","[Thabo Sefolosha, Hasheem Thabeet, Reggie Jackson, Jeremy Lamb, Nick Collison]",1,1
8,1,131,138,0:21,0:00,"[Melih Mahmutoglu, Izzet Turkyilmaz, Gasper Vidmar, James Birsen, Kenan Sipahi]","[Thabo Sefolosha, Hasheem Thabeet, Reggie Jackson, Jeremy Lamb, Nick Collison]",2,0
9,2,141,158,11:45,10:32,"[Melih Mahmutoglu, Izzet Turkyilmaz, Gasper Vidmar, James Birsen, Kenan Sipahi]","[Thabo Sefolosha, Hasheem Thabeet, Reggie Jackson, Jeremy Lamb, Nick Collison]",1,2


## Save the prototype stint dataset

The one-game stint dataset and substitution anomaly log are saved as interim artifacts for validation, summarization, and downstream lineup analysis.

At this stage, the stint dataset represents a stable prototype: lineup integrity is preserved across all stints, and substitution edge cases are tracked separately for transparency and future refinement.

The anomaly dataset provides a detailed record of substitution events that required tolerant handling, enabling targeted inspection and future improvements to the reconstruction pipeline.

In [82]:
# Save stint dataset
stints_df.to_pickle("../data/interim/stints_prototype_game.pkl")
stints_df.to_csv("../data/interim/stints_prototype_game.csv", index=False)

# Save substitution anomalies (NEW)
sub_anomalies_df.to_pickle("../data/interim/sub_anomalies_prototype_game.pkl")
sub_anomalies_df.to_csv("../data/interim/sub_anomalies_prototype_game.csv", index=False)

## Key findings

This notebook builds a working prototype of a stint-construction pipeline from NBA play-by-play data.

Using one game as a controlled test case, the notebook:

- standardized player and team identifiers
- parsed score progression into numeric fields
- extracted structured substitution events
- identified home and away teams
- built a player-name lookup using both reference-table and play-by-play sources
- initialized starting lineups
- tracked lineup changes over time
- segmented the game into lineup stints

After refining the substitution update logic, the resulting stint dataset preserves five-player lineup integrity across all generated stints. The prototype also logs substitution events that do not align perfectly with the inferred on-court state, which provides a transparent record of remaining data inconsistencies without breaking the lineup reconstruction.

This notebook establishes a valid foundation for downstream lineup analysis, while also documenting the edge cases that should be monitored in future iterations.

## Next step

With the stint-construction pipeline producing consistent five-player lineups, the next stage of the project is to validate and summarize the reconstructed dataset before applying performance metrics.

The focus shifts from data engineering to analytical validation. In the next notebook, we will:

- analyze substitution anomalies to quantify and contextualize remaining inconsistencies
- explore stint-level distributions, including duration and scoring patterns
- validate lineup continuity across periods and game phases
- assess whether the dataset is suitable for lineup and player impact modeling

This validation layer ensures that downstream metrics are built on a reliable representation of on-court lineup states.